In [65]:
import pandas as pd 
from glob import glob 
import sys 
sys.path.append('/home/work/yuna/HPA') 
from preprocessing.utils import MODELNAMES 
root_dir = '/home/work/yuna/HPA/evaluation/scored'

def find_matching(f, targets): 
    for t in targets : 
        if t in f : 
            f = f.replace(f'{t}', '')   
            return t, f 
    print(f"cannot find matching {f} in {targets}") 

def get_summary(dataset='mmstar'): 
    files = glob(f"{root_dir}/*/*{dataset}*.jsonl")  + glob(f"{root_dir}/*/*/*/{dataset}*.jsonl")

    dfs= []
    for f in files: 
        try: 
            df = pd.read_json(f, lines=True)
            if 'finetuned' in f : 
                df['model'] = f.split('/')[-2].replace('fold_0', '')
            else: 
                df['model'], f = find_matching(f, [model.split('/')[-1] for model in MODELNAMES])  
            df['condition'] = f.split('/')[-1][:-6].replace(f'_', ' ').replace('vqa 1k', '').replace(f'{dataset}', '').strip()
            dfs.append(df)
        except Exception as e: 
            print(e)
    df = pd.concat(dfs)
    df['correct'] = pd.to_numeric(df['correct'], errors='coerce')
    df['correct'] = (df['correct'] * 100).round(1)
    print(len(files) ) 

    pt = df.pivot_table(
        index=['model'],  
        columns=['condition'], 
        values=['correct'],
        aggfunc=['mean', 'count']
    )
    pt = pt.round(4)
    pt.to_csv(f"./tables/summary_{dataset}.csv")
    return df 

In [ ]:
!python /home/work/yuna/HPA/evaluation/score_humans.py --human_data_dir n20 --with_similarity 
!python /home/work/yuna/HPA/evaluation/score_results.py  --input_dir finetuned  
!python /home/work/yuna/HPA/evaluation/score_results.py  --input_dir pretrained --with_similarity 

In [66]:
model_results = {}
for ds in ['mmstar', 'spubench', 'vqa_5k', 'vqa_1k']: 
    model_results[ds]  = get_summary(ds) 

cannot find matching /home/work/yuna/HPA/evaluation/scored/pretrained/Qwen3-0.6B_mmstar.jsonl in ['InternVL3_5-8B', 'InternVL3_5-4B', 'InternVL3_5-2B', 'InternVL3_5-1B', 'Qwen3-VL-2B-Instruct', 'Qwen3-VL-4B-Instruct', 'Qwen3-VL-8B-Instruct', 'llava-v1.6-vicuna-7b-hf', 'llava-v1.6-mistral-7b-hf', 'llava-1.5-7b-hf']
cannot unpack non-iterable NoneType object
cannot find matching /home/work/yuna/HPA/evaluation/scored/pretrained/Qwen3-8B-Base_mmstar.jsonl in ['InternVL3_5-8B', 'InternVL3_5-4B', 'InternVL3_5-2B', 'InternVL3_5-1B', 'Qwen3-VL-2B-Instruct', 'Qwen3-VL-4B-Instruct', 'Qwen3-VL-8B-Instruct', 'llava-v1.6-vicuna-7b-hf', 'llava-v1.6-mistral-7b-hf', 'llava-1.5-7b-hf']
cannot unpack non-iterable NoneType object
72
cannot find matching /home/work/yuna/HPA/evaluation/scored/pretrained/Qwen3-4B_spubench.jsonl in ['InternVL3_5-8B', 'InternVL3_5-4B', 'InternVL3_5-2B', 'InternVL3_5-1B', 'Qwen3-VL-2B-Instruct', 'Qwen3-VL-4B-Instruct', 'Qwen3-VL-8B-Instruct', 'llava-v1.6-vicuna-7b-hf', 'llava-

In [ ]:
### VQA Questions 
human_vqa=pd.read_csv('/home/work/yuna/HPA/evaluation/scored/humans/human_vqa_per_question.csv')
human_vqa['model'] = "humans" 
human_vqa['condition'] = "inst blind" 
human_vqa.rename(columns={"mean_accuracy": 'correct', 'qid': 'question_id'}, inplace=True) 
human_vqa['correct'] = pd.to_numeric(human_vqa['correct'], errors='coerce')
human_vqa['correct'] = (human_vqa['correct'] * 100).round(1) 
qids = human_vqa.question_id.unique()
print(len(qids))
human_vqa.columns 

374


Index(['Unnamed: 0', 'question_id', 'answer_type', 'num_responses', 'answers',
       'confidences', 'gt_answers', 'visual_gt', 'correct', 'std_accuracy',
       'mean_confidence', 'std_confidence', 'accuracies',
       'mean_visual_similarity', 'std_visual_similarity',
       'visual_similarities', 'agreement', 'model', 'condition'],
      dtype='object')

In [69]:
model_vqa = model_results['vqa_1k'] 
model_vqa = model_vqa[model_vqa['question_id'].isin(qids)]
model_vqa.head()

,image_id,question_id,question_type,question,answers,multiple_choice_answer,answer_type,pid,output,answer_similarity,correct,model,condition
2,COCO_val2014_000000356421.jpg,356421011,what is the,Question: What is the boy's skateboard balanci...,"[{'answer': 'air', 'answer_confidence': 'no', ...",nothing,other,2,pole,0.242588,0.0,InternVL3_5-1B,
5,COCO_val2014_000000011241.jpg,11241003,is this,Question: Is this enough food for more than tw...,"[{'answer': 'yes', 'answer_confidence': 'yes',...",yes,yes/no,5,yes,1.000000,100.0,InternVL3_5-1B,
7,COCO_val2014_000000537701.jpg,537701022,does this,Question: Does this man's tie match the backgr...,"[{'answer': 'no', 'answer_confidence': 'yes', ...",no,yes/no,7,no,1.000000,100.0,InternVL3_5-1B,
12,COCO_val2014_000000135486.jpg,135486006,what are,Question: What are they carrying? Answer the q...,"[{'answer': 'kite', 'answer_confidence': 'yes'...",kite,other,12,kite,1.000000,100.0,InternVL3_5-1B,
13,COCO_val2014_000000305329.jpg,305329002,what color is,Question: What color is his helmet? Answer the...,"[{'answer': 'gray and yellow', 'answer_confide...",black,other,13,black,1.000000,100.0,InternVL3_5-1B,


In [75]:
vqa_human_comparison = pd.concat([model_vqa[model_vqa['question_id'].isin(qids)] , human_vqa])
# vqa = vqa_human_comparison.groupby(['model', 'condition']).mean(numeric_only=True)['correct'] 
vqa = vqa_human_comparison.pivot_table( 
    index=['model'], 
    columns=['condition'],  # , 'category', 'l2_category' 
    values=['correct'],
    aggfunc=['mean'] # , 'count' 
).round(1)
vqa['blind'] = vqa[('mean', 'correct', 'inst blind')] - vqa[('mean', 'correct', 'blind')]
vqa['MG'] = vqa[('mean', 'correct', '')] - vqa[('mean', 'correct', 'inst blind')] 
vqa.to_csv(f'./tables/vqa_human_comparison.csv')

In [88]:
vqa_flat = vqa.copy()
vqa_flat.columns = ['_'.join([str(i) for i in col if str(i) != '']).strip('_') for col in vqa_flat.columns.values]
df = vqa_flat.dropna(subset=['mean_correct', 'MG', 'blind'], axis=0).dropna(axis=1)
df.to_latex(f'./tables/A1_vqa_instruction.tex', float_format="%.1f")
df

,mean_correct,mean_correct_blind,mean_correct_inst blind,blind,MG
model,,,,,
InternVL3_5-1B,80.5,43.6,43.8,0.2,36.7
InternVL3_5-2B,82.1,45.5,45.3,-0.2,36.8
InternVL3_5-4B,83.0,45.0,47.2,2.2,35.8
InternVL3_5-8B,86.9,47.1,45.4,-1.7,41.5
Qwen3-VL-2B-Instruct,83.0,45.5,46.2,0.7,36.8
Qwen3-VL-4B-Instruct,85.9,45.5,47.4,1.9,38.5
Qwen3-VL-8B-Instruct,89.4,44.3,47.4,3.1,42.0
llava-v1.6-mistral-7b-hf,86.3,45.1,46.7,1.6,39.6


In [ ]:
vqa.dropna(axis=0).to_latex(f'./tables/vqa_human_comparison_pretrained.tex',
    float_format="%.1f")
vqa  

In [ ]:
### MMStar questions 
human_mc=pd.read_csv('/home/work/yuna/HPA/evaluation/scored/humans/human_mc_per_question.csv') 
qids = human_mc.pid.unique() 
model_mc = model_results['mmstar']
human_mc['pid'] = human_mc['pid'].astype('Int64')
model_mc['pid'] = model_mc['pid'].astype('Int64') 
print(len(qids))


In [41]:
pt = model_mc.pivot_table( 
    index=['model', 'pid', 'category', 'l2_category'],  
    columns=['condition'],  
    values=['correct'],
    aggfunc=['mean'] # , 'count' 
)
pt['MG'] = pt[('mean', 'correct', '')] - pt[('mean', 'correct', 'inst blind')] 
pt.to_csv(f'./tables/mmstar_model_MG_by_qid.csv')
pt.pivot_table( 
    index=['model'],  
    columns=['category', 'l2_category'],  
    values=['MG'],
    aggfunc=['mean'] # , 'count' 
).round(3).to_csv(f'./tables/mmstar_model_MG_by_category.csv')

In [47]:
mmstar_human_comparison = pd.concat([model_mc[model_mc['pid'].isin(qids)] , human_mc])
pt = mmstar_human_comparison.pivot_table( 
    index=['model'], 
    columns=['condition', 'category', 'l2_category'], 
    values=['correct'],
    aggfunc=['mean'] # , 'count' 
)
pt = pt.round(3)
pt.to_csv(f'./tables/mmstar_human_comparison.csv')
# with pd.option_context('display.float_format', '{:0.3f}'.format):
    # display(pt)  
mean_correct = mmstar_human_comparison.groupby(['model', 'condition'])['correct'].count()
mean_correct

model                                              condition 
InternVL3_5-1B                                                   247
                                                   blind         247
                                                   inst blind    247
InternVL3_5-2B                                                   247
                                                   blind         247
                                                                ... 
llava-v1.6-mistral-7b-hf_A4_mmstar_15_blind_inst   inst blind    247
llava-v1.6-mistral-7b-hf_SFT_mmstar_15_blind_inst                247
                                                   inst blind    247
llava-v1.6-mistral-7b-hf_SFT_vqa_15_blind_inst                   247
                                                   inst blind    247
Name: correct, Length: 69, dtype: int64

In [56]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams.update({
    'font.size': 11,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 14,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.linewidth': 1.2,
    'grid.linewidth': 0.5,
    'lines.linewidth': 2,
})

# Optional: Color palette
colors = sns.color_palette("colorblind")

In [ ]:
from scipy.stats import wasserstein_distance, ks_2samp

In [ ]:
def plot_vqa_distributions_with_metrics(H, M, model_name): 

    plt.figure(figsize=(6,4))
    sns.histplot(H, bins=20, kde=True, stat="density",
                 alpha=0.5, label="Human avg")
    sns.histplot(M, bins=20, kde=True, stat="density",
                 alpha=0.5, label="Model")

    plt.xlim(0,1)
    plt.xlabel("VQA score")
    plt.title(
        f"{model_name}\n"
        f"Wasserstein={wd:.3f}, KS={ks:.3f} (p={ks_p:.1e})"
    )
    plt.legend()
    plt.tight_layout()
    # plt.show()
    plt.savefig(f"./figures/distribution_{model_name}", dpi=300)  

    wd = wasserstein_distance(H, M)
    ks, ks_p = ks_2samp(H, M)

    return {
        "wasserstein": wd,
        "ks_stat": ks,
        "ks_p": ks_p,
        "human_mean": H.mean(),
        "model_mean": M.mean(),
    }


In [ ]:
models = vqa_human_comparison.model.unique() 
len(models)

28

In [3]:
pt = vqa_human_comparison.pivot_table(
    index=['model'], 
    columns=['condition'], 
    values=['correct'], # , "mean_confidence" 
    aggfunc=['mean', 'count']
)
pt = pt.round(4)
pt.to_csv(f'./tables/vqa_human_comparison.csv')
pt 

NameError: name 'vqa_human_comparison' is not defined

# MMStar 

In [44]:
dfs = []
for filepath in glob("/home/work/yuna/HPA/results/swift/*mmstar*.jsonl"): 
    print(f"Evaluating: {filepath}")
    df = read_file(filepath) 
    dfs.append(df)
    # results = evaluate_results(filepath)
    # print_report(results)

Evaluating: /home/work/yuna/HPA/results/swift/llava-v1.6-mistral-7b-hf_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_blind.jsonl
'output' cannot process /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-VL-4B-Instruct_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-2B_mmstar_sys_inst_blind.jsonl
'output' cannot process /home/work/yuna/HPA/results/swift/InternVL3_5-2B_mmstar_sys_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/llava-v1.6-mistral-7b-hf_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-VL-4B-Instruct_mmstar_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-0.6B_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/llava-v

In [54]:
df = pd.concat(dfs)
df = df[df['condition'] != '_blind']
results = df.groupby(['model_full', 'condition', 'category', 'l2_category'])['correct'].mean().reset_index()
results.pivot_table(index=['model_full', 'condition'], columns=[ 'category', 'l2_category'], values=['correct'])

correct  \
category                                      coarse perception   
l2_category                                       image emotion   
model_full                        condition                       
OpenGVLab/InternVL3_5-2B          _inst_blind          0.193548   
OpenGVLab/InternVL3_5-4B                               0.000000   
                                  _inst_blind          0.258065   
OpenGVLab/InternVL3_5-8B                               0.774194   
                                  _inst_blind          0.451613   
Qwen/Qwen3-VL-4B-Instruct                              0.161290   
                                  _inst_blind          0.322581   
Qwen/Qwen3-VL-8B-Instruct                              0.483871   
                                  _inst_blind          0.161290   
llava-hf/llava-v1.6-mistral-7b-hf                      0.580645   
                                  _inst_blind          0.032258   

                                                                     \
category                                                              
l2_category                                   image scene and topic   
model_full                        condition                           
OpenGVLab/InternVL3_5-2B          _inst_blind              0.042553   
OpenGVLab/InternVL3_5-4B                                   0.047619   
                                  _inst_blind              0.063830   
OpenGVLab/InternVL3_5-8B                                   0.588652   
                                  _inst_blind              0.283688   
Qwen/Qwen3-VL-4B-Instruct                                  0.411348   
                                  _inst_blind              0.148936   
Qwen/Qwen3-VL-8B-Instruct                                  0.425532   
                                  _inst_blind              0.198582   
llava-hf/llava-v1.6-mistral-7b-hf                          0.453901   
                                  _inst_blind              0.212766   

                                                                     \
category                                                              
l2_category                                   image style & quality   
model_full                        condition                           
OpenGVLab/InternVL3_5-2B          _inst_blind              0.038462   
OpenGVLab/InternVL3_5-4B                                   0.333333   
                                  _inst_blind              0.000000   
OpenGVLab/InternVL3_5-8B                                   0.769231   
                                  _inst_blind              0.333333   
Qwen/Qwen3-VL-4B-Instruct                                  0.397436   
                                  _inst_blind              0.153846   
Qwen/Qwen3-VL-8B-Instruct                                  0.666667   
                                  _inst_blind              0.141026   
llava-hf/llava-v1.6-mistral-7b-hf                          0.628205   
                                  _inst_blind              0.089744   

                                                                       \
category                                      fine-grained perception   
l2_category                                              localization   
model_full                        condition                             
OpenGVLab/InternVL3_5-2B          _inst_blind                   0.000   
OpenGVLab/InternVL3_5-4B                                          NaN   
                                  _inst_blind                   0.100   
OpenGVLab/InternVL3_5-8B                                        0.675   
                                  _inst_blind                   0.250   
Qwen/Qwen3-VL-4B-Instruct                                       0.325   
                                  _inst_blind                   0.150   
Qwen/Qwen3-VL-8B-Instruct                                       0.550   
                                  _inst_bl